# Experimento 4 — SVM com balanceamento utilizando todas as variáveis

Este experimento combina as duas principais estratégias avaliadas anteriormente:

- ampliação do conjunto de variáveis ambientais (Experimento 3);
- utilização de balanceamento de classes com `class_weight="balanced"` (Experimento 2).

O objetivo é verificar se a combinação entre maior quantidade de informações físico-químicas e penalização matemática das classes minoritárias melhora a capacidade do SVM em detectar situações críticas de qualidade da água.

As variáveis utilizadas foram:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `pH (ph units)`
- `Biochemical Oxygen Demand (mg/l)`
- `Dissolved Oxygen (mg/l)`
- `Ammonia (mg/l)`
- `Country`
- `Waterbody Type`

A variável alvo (`y`) foi:

- `conama_status`

O dataset foi dividido em:
- 80% treino
- 20% teste

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')

    DATA_PATH = Path(
        "/content/drive/MyDrive/dataset_pisi3/amostra_rotulada.parquet"
    )

else:
    print("Ambiente local/VS Code detectado.")

    DATA_PATH = Path(
        "C:/Users/anaju\OneDrive/projetos_lucas/AquaSense-ML/amostra_rotulada.parquet"
    )

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado.")
print(df.shape)

df.head()

Ambiente local/VS Code detectado.
Dataset carregado.
(141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("Bibliotecas de ML carregadas.")

Bibliotecas de ML carregadas.


In [4]:
X = df[
    [
        "Temperature (cel)",
        "Orthophosphate (mg/l)",
        "pH (ph units)",
        "Biochemical Oxygen Demand (mg/l)",
        "Dissolved Oxygen (mg/l)",
        "Ammonia (mg/l)",
        "Country",
        "Waterbody Type"
    ]
]

y = df["conama_status"]

print("X e y definidos.")

X e y definidos.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 8)
Teste: (28280, 8)


In [6]:
categorical_features = [
    "Country",
    "Waterbody Type"
]

numeric_features = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "pH (ph units)",
    "Biochemical Oxygen Demand (mg/l)",
    "Dissolved Oxygen (mg/l)",
    "Ammonia (mg/l)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            StandardScaler(),
            numeric_features
        )
    ]
)

print("Pré-processamento configurado.")

Pré-processamento configurado.


In [7]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            SVC(
                kernel="linear",
                class_weight="balanced",
                random_state=SEED
            )
        )
    ]
)

print("Treinando SVM balanceado...")

model.fit(X_train, y_train)

print("Treinamento concluído.")

Treinando SVM balanceado...
Treinamento concluído.


## Avaliação das Métricas de Treino

In [8]:
y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted",
    zero_division=0
)

print("Train Accuracy:")
print(train_accuracy)

print("\nTrain Precision:")
print(train_precision)

print("\nTrain Recall:")
print(train_recall)

print("\nTrain F1:")
print(train_f1)

print("\nTrain Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred))

Train Accuracy:
0.7707546919615627

Train Precision:
0.797435684788659

Train Recall:
0.7707546919615627

Train F1:
0.7779919497843673

Train Confusion Matrix:
[[ 4554   832  2133    41]
 [ 5578  7921  1469  6710]
 [  378    50   676     2]
 [ 1656  6297   786 74036]]


## Avaliação no Conjunto de Teste

In [9]:
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

test_precision = precision_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:")
print(test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.7733026874115984

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.39      0.60      0.47      1890
         Boa       0.53      0.37      0.44      5419
     Crítica       0.13      0.62      0.21       277
   Excelente       0.92      0.90      0.91     20694

    accuracy                           0.77     28280
   macro avg       0.49      0.62      0.51     28280
weighted avg       0.80      0.77      0.78     28280


Confusion Matrix:
[[ 1128   203   554     5]
 [ 1310  2006   403  1700]
 [   91    14   172     0]
 [  370  1555   206 18563]]


## Resultados Obtidos — Experimento 4

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.773
```

O Experimento 4 combinou as duas principais estratégias avaliadas anteriormente:

- ampliação das variáveis ambientais;
- balanceamento de classes com `class_weight="balanced"`.

O objetivo foi verificar se a combinação dessas abordagens permitiria ao SVM detectar melhor as classes minoritárias sem perder totalmente o desempenho global obtido no Experimento 3.



## Comparação entre algoritmos — Experimento 4

| Métrica | RF Exp. 4 | LightGBM Exp. 4 | Regressão Logística Exp. 4 | SVM Exp. 4 |
|---|---:|---:|---:|---:|
| Accuracy teste | 0.764 | 0.731 | 0.748 | 0.773 |
| Precision Atenção | 0.34 | 0.30 | 0.36 | 0.39 |
| Precision Boa | 0.47 | 0.42 | 0.49 | 0.53 |
| Precision Crítica | 0.10 | 0.09 | 0.11 | 0.13 |
| Precision Excelente | 0.90 | 0.88 | 0.91 | 0.92 |
| Recall Atenção | 0.55 | 0.51 | 0.57 | 0.60 |
| Recall Boa | 0.32 | 0.28 | 0.35 | 0.37 |
| Recall Crítica | 0.58 | 0.54 | 0.60 | 0.62 |
| Recall Excelente | 0.89 | 0.86 | 0.89 | 0.90 |
| F1 Atenção | 0.42 | 0.38 | 0.44 | 0.47 |
| F1 Boa | 0.38 | 0.33 | 0.41 | 0.44 |
| F1 Crítica | 0.17 | 0.15 | 0.19 | 0.21 |
| F1 Excelente | 0.89 | 0.87 | 0.90 | 0.91 |
| Macro avg F1 | 0.47 | 0.43 | 0.49 | 0.51 |
| Weighted avg F1 | 0.76 | 0.72 | 0.75 | 0.78 |



## Análise das métricas

O Experimento 4 apresentou um dos resultados mais equilibrados entre todos os experimentos realizados no projeto AquaSense até o momento.

O SVM balanceado com ampliação das variáveis ambientais atingiu:
- accuracy de aproximadamente 0.773;
- weighted F1-score de 0.78;
- macro avg F1 de 0.51.

Apesar da acurácia ter ficado abaixo do Experimento 3 (0.829), houve melhora extremamente importante na detecção das classes minoritárias, especialmente da classe `Crítica`.

O recall da classe `Crítica` atingiu 0.62, representando o melhor resultado obtido entre todos os algoritmos avaliados neste experimento:
- Random Forest: 0.58;
- LightGBM: 0.54;
- Regressão Logística: 0.60;
- SVM: 0.62.

Esse resultado demonstra que o SVM conseguiu identificar corretamente a maior parte das amostras críticas, algo que não ocorria nos experimentos sem balanceamento.

Além disso:
- a classe `Atenção` apresentou recall de 0.60;
- a classe `Boa` apresentou recall de 0.37;
- a classe `Excelente` manteve desempenho elevado com recall de 0.90.

Isso mostra que o modelo conseguiu melhorar significativamente a distribuição das previsões entre as classes.

A precision da classe `Crítica` permaneceu baixa (0.13), indicando que ainda ocorreram muitos falsos positivos. Entretanto, em problemas ambientais, o recall costuma ser mais importante do que a precision para classes críticas, pois é preferível gerar alguns falsos alertas do que ignorar situações potencialmente perigosas.

O SVM apresentou os melhores resultados gerais do Experimento 4 em praticamente todas as métricas:
- maior accuracy;
- maior weighted F1;
- maior macro avg F1;
- maior recall para `Crítica`;
- maior F1-score para `Crítica`.



## Análise da Matriz de Confusão

A matriz de confusão mostra que o modelo passou a distribuir melhor suas previsões entre todas as classes.

O SVM conseguiu identificar corretamente:
- 1128 amostras da classe `Atenção`;
- 2006 amostras da classe `Boa`;
- 172 amostras da classe `Crítica`;
- 18563 amostras da classe `Excelente`.

Comparado aos experimentos anteriores:
- houve forte redução do colapso na classe majoritária;
- a classe `Crítica` finalmente passou a ser detectada de forma consistente;
- a distribuição das previsões tornou-se mais equilibrada.

Entretanto, ainda houve confusão importante entre:
- `Boa` e `Excelente`;
- `Atenção` e `Boa`.

Isso indica que as fronteiras entre classes intermediárias continuam complexas mesmo após o balanceamento.



## Conclusão

O Experimento 4 representou o melhor equilíbrio obtido até o momento entre:
- desempenho global;
- estabilidade;
- detecção de classes críticas.

A combinação entre:
- ampliação das variáveis ambientais;
- e balanceamento de classes;

foi extremamente importante para melhorar a capacidade do SVM em detectar amostras ambientalmente relevantes.

Embora o Experimento 3 tenha apresentado maior acurácia geral, ele ignorava completamente a classe `Crítica`. Já o Experimento 4 conseguiu manter boa performance global enquanto aumentava drasticamente o recall das classes minoritárias.

Entre todos os algoritmos comparados neste experimento, o SVM apresentou:
- melhor recall para `Crítica`;
- melhor F1-score para `Crítica`;
- melhor weighted F1-score;
- melhor macro avg F1;
- melhor equilíbrio geral entre as métricas.

Os resultados sugerem que:
- a ampliação das variáveis melhora a separação das classes;
- o balanceamento reduz o viés da classe majoritária;
- e a combinação dessas estratégias produz os melhores resultados para o contexto ambiental do AquaSense.

Como continuidade metodológica, os próximos experimentos poderão explorar:
- tuning de hiperparâmetros;
- técnicas avançadas de balanceamento;
- ensemble methods;
- otimização específica para classes raras.

Essas abordagens podem melhorar ainda mais a detecção de amostras críticas sem comprometer o desempenho global do modelo.